In [1]:
import gradio as gr
import pandas as pd
import torch
import torch.nn as nn
import json
import os
import numpy as np
from catboost import CatBoostClassifier, CatBoostRegressor, Pool

# --- 1. АРХИТЕКТУРА МОДЕЛИ BERT ---
class TransformerRec(nn.Module):
    def __init__(self, vocab_size, hidden=256, heads=8, layers=4, max_len=20, dropout=0.2):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, hidden)
        self.pos_emb = nn.Embedding(max_len, hidden)
        self.dropout = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden, nhead=heads, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        x = self.dropout(self.token_emb(x) + self.pos_emb(positions))
        x = self.transformer(x)
        return self.fc(x)

# --- 2. НАСТРОЙКИ И ЗАГРУЗКА ДАННЫХ ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
data_folder = "../gym_model"

LEVEL_MAP = {'Novice': 0, 'Beginner': 1, 'Intermediate': 2, 'Advanced': 3}
GOAL_MAP = {
    'Powerlifting': {'reps': 3, 'int': 0.88},
    'Powerbuilding': {'reps': 8, 'int': 0.78},
    'Bodybuilding': {'reps': 10, 'int': 0.72},
    'Athletics': {'reps': 12, 'int': 0.60},
    'Muscle & Sculpting': {'reps': 14, 'int': 0.55},
    'Fitness': {'reps': 12, 'int': 0.60},
    'Bodyweight Fitness': {'reps': 12, 'int': 0.60}
}
INVENTORY_MAP = {
    'Bands': 'Bodyweight', 'Cardio': 'Bodyweight', 'Bodyweight': 'Bodyweight',
    'Machine': 'Machine', 'Dumbbell': 'Dumbbell', 'Barbell': 'Barbell'
}

with open(f'{data_folder}/vocab.json', 'r') as f:
    vocab = json.load(f)
id2name = {v: k for k, v in vocab.items()}

df_meta = pd.read_csv("../programs_detailed_boostcamp_kaggle.csv")
exercise_meta_dict = df_meta.drop_duplicates(subset=['exercise_name']).set_index('exercise_name').to_dict('index')

df_equip = pd.read_csv("../catboost-classifier/datasets/id_to_equipment_mapping_FIXED.csv")
id_to_eq_dict = df_equip.set_index('candidate_id')['equipment_golden'].to_dict()

df_ex_base = pd.read_csv('./datasets/exercises_base_FINAL_CLEANED.csv')
ex_to_base_dict = df_ex_base.set_index('Exercise_Name')['Base_Lift'].to_dict()

bert_model = TransformerRec(vocab_size=len(vocab)).to(device)
bert_path = f'{data_folder}/checkpoints-2/gym_bert_v2_ep27_hit0.3522.pth'
if os.path.exists(bert_path):
    bert_model.load_state_dict(torch.load(bert_path, map_location=device))
    bert_model.eval()

cat_ranker = CatBoostClassifier()
cat_ranker.load_model(f"{data_folder}/catboost_recommender_final.cbm")

weight_predictor_light = CatBoostRegressor()
weight_predictor_light.load_model("weight_predictor_light.cbm")

weight_predictor_pro = CatBoostRegressor()
weight_predictor_pro.load_model("weight_predictor_pro.cbm")

# --- 3. ИНТЕЛЛЕКТУАЛЬНЫЙ РАСЧЕТ ВЕСА (ТРЕНЕРСКАЯ ЛОГИКА) ---
def get_weight_recommendation(ex_name, user_profile):
    has_records = sum(user_profile['sbd']) > 0
    raw_eq = id_to_eq_dict.get(vocab.get(ex_name), 'Machine')
    eq_clean = INVENTORY_MAP.get(raw_eq, 'Machine').lower()
    base_lift = ex_to_base_dict.get(ex_name, 'Best3DeadliftKg')
    ex_name_l = ex_name.lower()
    
    if has_records:
        cols = ['Sex', 'Age', 'BodyweightKg', 'Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg', 'goal_clean', 'level_idx', 'Base_Lift', 'eq_clean', 'exercise_id']
        input_data = {
            'Sex': str(user_profile['sex']), 'Age': float(user_profile['age']),
            'BodyweightKg': float(user_profile['bw']),
            'Best3SquatKg': float(user_profile['sbd'][0]), 'Best3BenchKg': float(user_profile['sbd'][1]), 'Best3DeadliftKg': float(user_profile['sbd'][2]),
            'goal_clean': str(user_profile['goal']), 'level_idx': int(LEVEL_MAP.get(user_profile['level'], 1)),
            'Base_Lift': str(base_lift), 'eq_clean': str(INVENTORY_MAP.get(raw_eq, 'Machine')), 'exercise_id': str(vocab.get(ex_name))
        }
        model = weight_predictor_pro
        cat_features = ['Sex', 'goal_clean', 'Base_Lift', 'eq_clean', 'exercise_id']
    else:
        cols = ['Sex', 'Age', 'BodyweightKg', 'goal_clean', 'level_idx', 'eq_clean', 'exercise_id']
        input_data = {
            'Sex': str(user_profile['sex']), 'Age': float(user_profile['age']), 'BodyweightKg': float(user_profile['bw']),
            'goal_clean': str(user_profile['goal']), 'level_idx': int(LEVEL_MAP.get(user_profile['level'], 1)),
            'eq_clean': str(INVENTORY_MAP.get(raw_eq, 'Machine')), 'exercise_id': str(vocab.get(ex_name))
        }
        model = weight_predictor_light
        cat_features = ['Sex', 'goal_clean', 'eq_clean', 'exercise_id']

    input_df = pd.DataFrame([input_data])[cols]
    pred = model.predict(Pool(input_df, cat_features=cat_features))[0]
    
    weight, reps = pred[0], int(pred[1])
    
    if 'dumbbell' in eq_clean:
        weight = max(2.0, round((weight) / 2) * 2)
    elif 'machine' in eq_clean:
        weight = max(5.0, round(weight / 5) * 5)
    elif 'barbell' in eq_clean:
        is_base = any(b in ex_name_l for b in ['bench', 'squat', 'deadlift'])
        weight = max(20.0 if is_base else 5.0, round(weight / 2.5) * 2.5)
    
    if any(kw in ex_name_l for kw in ['plank', 'run', 'skip', 'jump', 'stretch']):
        weight = 0.0

    # ФИКС: Отрицательные значения -> 0.0
    final_weight = round(max(0.0, weight), 1)

    return final_weight, reps, ("PRO" if has_records else "LIGHT")

# --- 4. ОСНОВНАЯ ЛОГИКА ПРЕДСКАЗАНИЯ ---
def predict_debug(history_ids, user_profile, top_k=10):
    input_seq = torch.LongTensor([history_ids]).to(device)
    with torch.no_grad():
        logits = bert_model(input_seq)
        probs = torch.softmax(logits[0, -1, :], dim=-1)
    
    scores, indices = torch.topk(probs, k=100)
    candidates = []
    for s, idx in zip(scores.cpu().numpy(), indices.cpu().numpy()):
        idx_val = int(idx)
        name = id2name[idx_val]
        raw_eq = id_to_eq_dict.get(idx_val, 'Unknown')
        candidates.append({
            'bert_score': float(s), 'candidate_id': str(idx_val),
            'is_same_group': 1 if exercise_meta_dict.get(name, {}).get('category') == exercise_meta_dict.get(id2name[history_ids[-1]], {}).get('category') else 0,
            'eq_clean': str(INVENTORY_MAP.get(raw_eq, raw_eq)),
            'level_idx': LEVEL_MAP.get(user_profile['level'], 1), 'goal_clean': str(user_profile['goal'])
        })
    
    cand_df = pd.DataFrame(candidates)
    preds = cat_ranker.predict_proba(cand_df[['bert_score', 'candidate_id', 'is_same_group', 'eq_clean', 'level_idx', 'goal_clean']])[:, 1]
    cand_df['final_score'] = preds
    cand_df['name'] = [id2name[int(i)] for i in cand_df['candidate_id']]
    
    if user_profile['equipment'] != 'All (Gym Mixed)':
        cand_df = cand_df[cand_df['eq_clean'] == user_profile['equipment']]
    
    final_df = cand_df.sort_values('final_score', ascending=False).head(top_k)
    weights_info = [get_weight_recommendation(n, user_profile) for n in final_df['name']]
    
    cat_table = pd.DataFrame({
        'Упражнение': final_df['name'].values,
        'Вес (кг)': [x[0] for x in weights_info],
        'Повторы': [x[1] for x in weights_info],
        'Инвентарь': final_df['eq_clean'].values,
        'Движок ИИ': [x[2] for x in weights_info],
        'Score': final_df['final_score'].values
    })
    
    return final_df['name'].tolist(), cat_table

# --- 5. ИНТЕРФЕЙС GRADIO ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    history_state = gr.State([])
    profile_state = gr.State({})
    
    gr.Markdown("# 🏋️‍♂️ AI Trainer: Hybrid Intelligence")
    
    with gr.Row(variant='panel'):
        with gr.Column():
            sex = gr.Radio(['M', 'F', 'Mx'], label="Пол", value="M")
            age = gr.Number(label="Возраст", value=25)
            bw = gr.Number(label="Ваш вес (кг)", value=80)
            level = gr.Dropdown(['Novice', 'Beginner', 'Intermediate', 'Advanced'], label="Уровень", value="Intermediate")
            goal = gr.Dropdown(list(GOAL_MAP.keys()), label="Цель", value="Bodybuilding")
            equip = gr.Dropdown(['Machine', 'Dumbbell', 'Barbell', 'Bodyweight', 'All (Gym Mixed)'], label="Инвентарь", value="All (Gym Mixed)")
        with gr.Column():
            gr.Markdown("### 🏆 Рекорды SBD")
            sq_max = gr.Number(label="Присед (кг)", value=0)
            bp_max = gr.Number(label="Жим (кг)", value=0)
            dl_max = gr.Number(label="Тяга (кг)", value=0)
    
    start_btn = gr.Button("🚀 Начать тренировку", variant="primary")

    with gr.Column(visible=False) as main_app:
        history_display = gr.Textbox(label="Цепочка упражнений", interactive=False)
        first_ex_drop = gr.Dropdown(choices=sorted(list(vocab.keys())), label="Первое упражнение:")
        results_row = gr.Row(visible=False)
        with results_row:
            cat_ui_table = gr.DataFrame()
        recommendations = gr.Dropdown(choices=[], label="Следующее упражнение:", visible=False)
        add_btn = gr.Button("➕ Добавить", variant="primary", visible=False)

    def start_fn(s, a, b, l, g, e, sm, bm, dm):
        profile = {'sex': s, 'age': a, 'bw': b, 'level': l, 'goal': g, 'equipment': e, 'sbd': [sm, bm, dm]}
        return {profile_state: profile, main_app: gr.update(visible=True)}

    def add_ex_fn(ex_name, history, profile):
        if not ex_name: return history, " ➔ ".join(history), gr.update(), gr.update(), gr.update(), gr.update()
        w, r, m_type = get_weight_recommendation(ex_name, profile)
        history.append(f"{ex_name} [{w}кг x {r}] ({m_type})")
        history_ids = [vocab[name.split(" [")[0]] for name in history]
        recs, c_df = predict_debug(history_ids, profile)
        return history, " ➔ ".join(history), gr.update(choices=recs, value=None, visible=True), c_df, gr.update(visible=True), gr.update(visible=True)

    start_btn.click(start_fn, [sex, age, bw, level, goal, equip, sq_max, bp_max, dl_max], [profile_state, main_app])
    first_ex_drop.change(add_ex_fn, [first_ex_drop, history_state, profile_state], [history_state, history_display, recommendations, cat_ui_table, results_row, add_btn])
    add_btn.click(add_ex_fn, [recommendations, history_state, profile_state], [history_state, history_display, recommendations, cat_ui_table, results_row, add_btn])

demo.launch()

/Users/artemmarkov/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/artemmarkov/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
